In [ ]:
import queue
import sounddevice as sd
from vosk import Model, KaldiRecognizer
from pynput import keyboard
import json
import os

# 設定
SAMPLERATE = 16000
MODEL_PATH = "model_standard"

# 共有キューと状態管理
q = queue.Queue()
recording = False

def callback(indata, frames, time, status):
    """マイクから入力された音声データをキューに入れる"""
    if recording:
        q.put(bytes(indata))

def on_press(key):
    global recording
    if key == keyboard.Key.space: # スペースキーで録音ON/OFF
        recording = not recording
        if recording:
            print("\n[REC START]")
        else:
            print("\n[REC STOP]")
        if key == keyboard.Key.esc:
            print("プログラムを終了します...")
            os._exit(0) # 強制終了

# モデルと認識器の準備
model = Model(MODEL_PATH)
rec = KaldiRecognizer(model, SAMPLERATE)

# キーボード監視の開始
listener = keyboard.Listener(on_press=on_press)
listener.start()

print(f"Spaceキーで録音の開始/停止を切り替えます (Sampling Rate: {SAMPLERATE}Hz)")

try:
    # 音声入力ストリームの開始
    with sd.RawInputStream(samplerate=SAMPLERATE, blocksize=8000, dtype='int16',
                           channels=1, callback=callback):
        while True:
            data = q.get()
            if rec.AcceptWaveform(data):
                # 確定結果 (JSON形式なのでテキスト部分だけ抽出)
                result = json.loads(rec.Result())
                print(f"確定: {result['text']}")
            else:
                # 途中経過
                partial = json.loads(rec.PartialResult())
                if partial['partial']:
                    print(f"途中: {partial['partial']}", end='\r')

except KeyboardInterrupt:
    print("\n終了します")



[REC START]
確定: ちはやぶる 神代 も 聞か ず 竜田
確定: 紅 に 水

[REC STOP]

[REC START]
確定: 
確定: ちはやぶる 神代 も 大きかっ た
確定: 発十
確定: から 紅 に 水
確定: 

[REC STOP]

[REC START]
確定: ちはやぶる 神代 も 聞か ず 発
確定: あらー
確定: 紅 に 水
確定: 

[REC STOP]

[REC START]
確定: ちはやぶる 神代 も きか ず 竜田 川 から 紅 に 水 くる とい に 導く くる と

[REC STOP]


In [ ]:
import queue
import sounddevice as sd
from vosk import Model, KaldiRecognizer
from pynput import keyboard
import json
import os
import csv
import sys

# --- 設定 ---
SAMPLERATE = 16000
MODEL_PATH = "model_standard"
KARUTA_DB = "data/karuta_db.csv"

q = queue.Queue()
recording = False

def load_karuta_grammar():
    """CSVからかるたの句を読み込み、Vosk用辞書文字列を作成する"""
    if not os.path.exists(KARUTA_DB):
        print(f"【ERROR】データベースファイルが見つかりません: {KARUTA_DB}")
        return None

    words_set = set()
    try:
        with open(KARUTA_DB, encoding="utf-8") as f:
            reader = csv.reader(f)
            for row in reader:
                # row[1]: 上の句, row[2]: 下の句 を想定
                if len(row) < 3:
                    continue
                
                upper = row[1].strip()
                lower = row[2].strip()

                if upper:
                    words_set.add(upper)        # 一文まるごと
                    for w in upper.split():      # スペース区切りがあれば単語ごと
                        if w: words_set.add(w)
                
                if lower:
                    words_set.add(lower)
                    for w in lower.split():
                        if w: words_set.add(w)

        if not words_set:
            print("【ERROR】CSVから有効なワードを抽出できませんでした。")
            return None

        # [重要] 辞書外の言葉を一切許容しないため、[unk] は入れない
        grammar_list = list(words_set)
        return json.dumps(grammar_list, ensure_ascii=False)

    except Exception as e:
        print(f"【ERROR】CSV読み込み中にエラー発生: {e}")
        return None

# --- 初期化 ---
print("--- 初期化中 ---")
if not os.path.exists(MODEL_PATH):
    print(f"【ERROR】Voskモデルが見つかりません: {MODEL_PATH}")
    sys.exit(1)

model = Model(MODEL_PATH)
grammar = load_karuta_grammar()

if grammar is None:
    print("辞書の作成に失敗したため、プログラムを終了します。")
    sys.exit(1)

# 認識器の作成 (grammarを第3引数に確実に渡す)
rec = KaldiRecognizer(model, SAMPLERATE, grammar)
print(f"[Success] 辞書モード起動: {len(json.loads(grammar))} 語を登録しました。")

# --- キーボード & 録音制御 ---
def callback(indata, frames, time, status):
    if status:
        print(f"Audio Error: {status}", file=sys.stderr)
    if recording:
        q.put(bytes(indata))

def on_press(key):
    global recording
    try:
        if key == keyboard.Key.space:
            recording = not recording
            print(f"\n--- {'録音開始 (REC ON)' if recording else '録音停止 (REC OFF)'} ---")
        
        if key == keyboard.Key.esc:
            print("\n[ESC] プログラムを安全に終了します...")
            os._exit(0)
    except Exception:
        pass

listener = keyboard.Listener(on_press=on_press)
listener.start()

print("\n操作方法:")
print("  [Space] : 録音の開始／停止")
print("  [Esc]   : プログラムの終了")
print("-" * 30)

# --- メインループ ---
try:
    with sd.RawInputStream(samplerate=SAMPLERATE, blocksize=4000, dtype='int16',
                           channels=1, callback=callback):
        while True:
            # 録音中でない時もループを回すが、データがなければ待機
            data = q.get()
            
            if rec.AcceptWaveform(data):
                res = json.loads(rec.Result())
                if res['text']:
                    print(f"\n【確定結果】: {res['text']}")
            else:
                part = json.loads(rec.PartialResult())
                if part['partial']:
                    # リアルタイムに推測中の文字を表示
                    print(f"推測中: {part['partial']}", end='\r')

except Exception as e:
    print(f"\n実行エラー: {e}")
finally:
    print("\n終了")